In [1]:
# See https://docs.scipy.org/doc/numpy/user/basics.subclassing.html

In [2]:
%matplotlib notebook
import sys, os
sys.path.append(os.path.abspath("../"))
import sam

In [3]:
from poissolve.mesh import EpiStack,Mesh

In [4]:
xp=5
xn=5
m=Mesh(EpiStack(['pGaN','GaN',5],['nGaN','GaN',5]),max_dz=1)#,refinements=[[xp,.1,1.3]])
sm=m.submesh([3,8])

In [5]:
import numpy as np

class Function(np.ndarray):

    def __new__(cls,*args,**kwargs):
        raise NotImplementedError
    
    def __array_finalize__(self, obj):
        if obj is None: return
        
        self.mesh = getattr(obj,'mesh',"View casting from ndarray not supported.")
    
    def restrict(self,submesh):
        # doesn't check that submesh and mesh are compatible
        return type(self)(submesh,self[submesh._slice])
    
class PointFunction(Function):

    def __new__(cls, mesh, value=np.NaN, dtype='float'):
        if hasattr(value,'__iter__'):
            obj = np.asarray(value).view(cls)
            assert obj.shape==mesh.z.shape,\
                "Given arr is not compatible with given mesh"
        else:
            obj = np.full(mesh.z.shape,value,dtype=dtype).view(cls)
        obj.mesh = mesh
        return obj
    
    #def __array_prepare__(self, out_arr, context=None):
    #    assert out_arr.shape==self.mesh.z.shape,\
    #        "Can't combine Functions of different mesh sizes"
    #    out_arr.shape=self.mesh.z.shape
    #    return out_arr
    
    def differentiate(self):
        return MidFunction(self.mesh,np.diff(self)/self.mesh._dz)
    
    def integrate(self,flipped=False):
        return np.cumsum(
            (self*self.mesh._dzp)[:-1]
                if not flipped
                else np.flipud(-self*self.mesh._dzp)[:-1]).view(MidFunction)
    
    
class MidFunction(Function):

    def __new__(cls, mesh, value=np.NaN, dtype='float'):
        if hasattr(value,'__iter__'):
            obj = np.asarray(value).view(cls)
            assert obj.shape==mesh.zp.shape,\
                "Given arr is not compatible with given mesh"
        else:
            obj = np.full(mesh.zp.shape,value,dtype=dtype).view(cls)
        obj.mesh = mesh
        return obj
    
    #def __array_prepare__(self, out_arr, context=None):
    #    assert out_arr.shape==self.mesh.zp.shape,\
    #        "Can't combine Functions of different mesh sizes"
    #    out_arr.shape=self.mesh.zp.shape
    #    return out_arr

    def differentiate(self,fill_value=np.NaN):
        pf=PointFunction(self.mesh)
        pf[1:-1]=np.diff(self)/self.mesh._dzp[1:-1]
        pf[[0,-1]]=fill_value
        return pf
    
    def integrate(self,flipped=False):
        #if output is None:
        output=PointFunction(self.mesh,value=0.0)
        np.cumsum(
            self*self.mesh._dz if not flipped else np.flipud(-self*self.mesh._dz),
            out=(output[1:] if not flipped else np.flipud(output[:-1])))
        return output
    
    def to_point_function(self,interp='unweighted'):
        if interp=='unweighted':
            arr=np.empty(len(self)+1)
            arr[1:-1]=(self[1:]+self[:-1])/2
            arr[[0,-1]]=arr[[1,-2]]
        if interp=='z':
            arr=interp1d(self._z,self.arr,
                fill_value='extrapolate')(self.mesh.z)
        return PointFunction(self.mesh,arr)
    
def MaterialFunction(mesh,prop,pos='mid'):
    # could make this more efficient by directly interpolating if Point case?
    # this function almost duplicates RegionFunction...

    ptcounts=np.diff([0]+[i for i,ll,lr in mesh.interfaces_point]+[len(mesh.z)-1])
    arr=[]

    propfunc=(lambda i: prop(mesh._layers[i].material))\
        if callable(prop)\
        else (lambda i: mesh._layers[i][prop])
    for i,ptc in enumerate(ptcounts):
        arr+=[propfunc(i)]*ptc
        
    out=MidFunction(mesh,arr)
    if pos=="point":
        return out.to_point_function()
    else: return out
    
def RegionFunction(mesh,prop,pos='mid'):
    # could make this more efficient by directly interpolating if Point case?

    ptcounts=np.diff([0]+[i for i,ll,lr in mesh.interfaces_point]+[len(mesh.z)-1])
    arr=[]

    propfunc=(lambda i: prop(mesh._layers[i].name))\
        if callable(prop)\
        else (lambda i: mesh._layers[i][prop])
    for i,ptc in enumerate(ptcounts):
        arr+=[propfunc(i)]*ptc
        
    out=MidFunction(mesh,arr)
    if pos=="point":
        return out.to_point_function()
    else: return out

In [6]:
pf=PointFunction(m,value=np.arange(11))
pf

PointFunction([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [7]:
spf=pf.restrict(sm)
print(spf)

[3 4 5 6 7]


In [8]:
spf[:]=0

In [9]:
print(pf)

[ 0  1  2  0  0  0  0  0  8  9 10]


In [10]:
pf.integrate().differentiate()

PointFunction([ nan,   1.,   2.,   0.,   0.,   0.,   0.,   0.,   8.,   9.,  nan])

In [11]:
pf.differentiate().integrate(flipped=True)

PointFunction([-10.,  -9.,  -8., -10., -10., -10., -10., -10.,  -2.,  -1.,   0.])

In [12]:
MidFunction(m,m._dz).to_point_function()

PointFunction([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.])

In [13]:
MaterialFunction(m,"Eg").to_point_function()

PointFunction([ 3.605,  3.605,  3.605,  3.605,  3.605,  3.605,  3.605,  3.605,
        3.605,  3.605,  3.605])

In [14]:
pf=PointFunction(m,value=1.3)*PointFunction(m,value=2)
print(pf)

[ 2.6  2.6  2.6  2.6  2.6  2.6  2.6  2.6  2.6  2.6  2.6]


In [15]:
pf+=2
pf/=2
print(pf)

[ 2.3  2.3  2.3  2.3  2.3  2.3  2.3  2.3  2.3  2.3  2.3]


In [16]:
pf.mesh

In [17]:
try:
    pf2=PointFunction(m,value=range(10))
except Exception as e:
    print(e)
    print('Trying a new one')
    pf2=PointFunction(m,value=range(11))
print(pf2)

Given arr is not compatible with given mesh
Trying a new one
[ 0  1  2  3  4  5  6  7  8  9 10]


In [18]:
mf=MidFunction(m,value=2)
print(mf)

[ 2.  2.  2.  2.  2.  2.  2.  2.  2.  2.]


In [19]:
mf+pf

ValueError: operands could not be broadcast together with shapes (10,) (11,) 

In [20]:
np.exp(np.NaN)

nan

In [21]:
from poissolve.maths.fermi_dirac_integral import fd12
fd12(12)

ImportError: /home/sam/PycharmProjects/Poissolve/poissolve/maths/fermi_dirac_integral.cpython-35m-x86_64-linux-gnu.so: undefined symbol: PyFPE_jbuf